# Namma Space — train a photoreal Gaussian splat of the hostel

This turns the hostel PoseCam capture into a **photoreal Gaussian splat** (like the home/kitchen scene).
Training needs an NVIDIA GPU, which free Colab provides.

**Before you start:** *Runtime → Change runtime type → **T4 GPU** → Save.* Then *Runtime → Run all*.
Step 1 stops immediately if the runtime has no GPU — training cannot run without one.

What happens (≈ 45–75 min in total):
1. installs Nerfstudio 1.1.5 (the first training step also compiles the GPU rasteriser — several minutes, normal);
2. downloads the prepared dataset from the repository: 1,376 PoseCam frames with ARCore poses (no COLMAP needed) and a 300k-point starting cloud;
3. trains `splatfacto` for 30,000 steps in the hostel's own coordinate frame (metres, floor at y = 0);
4. measures quality on 137 held-out photos (PSNR);
5. exports `splat.ply` and copies it to your Google Drive → share that file's link with the team.

Nothing else needs to be set: the dataset was built with `python -m navkit nerfstudio` and checked against Nerfstudio's own loader.

In [ ]:
# 1. Check the GPU. If this stops with an error: Runtime -> Change runtime type -> "T4 GPU" -> Save,
#    then Runtime -> Run all again. (If Colab says no GPU is available, your free GPU quota is used up
#    for now: try again later or from another Google account.)
import subprocess
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
if gpu.returncode != 0 or not gpu.stdout.strip():
    raise SystemExit('NO GPU in this runtime. Runtime -> Change runtime type -> T4 GPU -> Save, then Run all again.')
print('GPU:', gpu.stdout.strip())

In [ ]:
# 2. Install Nerfstudio (pins gsplat 1.4.0). The protobuf "dependency conflicts" warnings are harmless here.
!pip install -q nerfstudio==1.1.5
import torch
assert torch.cuda.is_available(), 'PyTorch cannot see the GPU - check step 1'
print('torch', torch.__version__, 'CUDA', torch.version.cuda, 'GPU', torch.cuda.get_device_name(0))

# Compile and test gsplat's GPU rasteriser now (takes several minutes the first time), so a problem
# shows up here instead of after the dataset download.
import gsplat
means = torch.randn(100, 3, device='cuda') + torch.tensor([0, 0, 3.0], device='cuda')
quats = torch.nn.functional.normalize(torch.randn(100, 4, device='cuda'), dim=-1)
img, alpha, _ = gsplat.rasterization(means, quats, torch.full((100, 3), 0.05, device='cuda'),
    torch.ones(100, device='cuda'), torch.rand(100, 3, device='cuda'), torch.eye(4, device='cuda')[None],
    torch.tensor([[[300.0, 0, 160], [0, 300.0, 120], [0, 0, 1]]], device='cuda'), 320, 240)
print('gsplat GPU rasteriser OK:', tuple(img.shape))

In [ ]:
# 3. Download the prepared hostel dataset (~45 MB)
!wget -q -O /content/hostel.zip https://github.com/atharv-namdeo/3d-mapping/raw/main/datasets/hostel_posecam_nerfstudio.zip
!unzip -q -o /content/hostel.zip -d /content/
!ls /content/hostel_posecam_nerfstudio && ls /content/hostel_posecam_nerfstudio/images | wc -l

### 4. Train (≈ 30–60 min on a T4)
The four `nerfstudio-data` flags keep PoseCam's metric, gravity-aligned frame so the result lines up with the
hostel's map, labels and routes with no manual alignment. Keep this tab open while it trains.

In [ ]:
# 4. Train. Output is long; the last lines are shown if anything goes wrong.
import glob, os
cmd = ['ns-train', 'splatfacto',
       '--data', '/content/hostel_posecam_nerfstudio',
       '--output-dir', '/content/outputs', '--experiment-name', 'hostel',
       '--max-num-iterations', '30000', '--vis', 'tensorboard',
       '--viewer.quit-on-train-completion', 'True',
       'nerfstudio-data', '--orientation-method', 'none', '--center-method', 'none',
       '--auto-scale-poses', 'False', '--load-3D-points', 'True']
with open('/content/train.log', 'w') as log:
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        log.write(line)
        if 'Step' in line or '%' in line or 'Error' in line or 'error' in line:
            print(line.rstrip()[-160:])
    proc.wait()
ckpts = glob.glob('/content/outputs/hostel/splatfacto/*/nerfstudio_models/*.ckpt')
if proc.returncode != 0 or not ckpts:
    print(open('/content/train.log').read()[-4000:])
    raise SystemExit('Training failed (see the log above). Send the last lines to the team.')
config = os.path.join(os.path.dirname(os.path.dirname(sorted(ckpts)[-1])), 'config.yml')
print('Trained:', sorted(ckpts)[-1])

In [ ]:
# 5. Quality on the 137 held-out photos (higher PSNR = sharper; > 22 dB is good for 640x480 phone frames)
import json
ev = subprocess.run(['ns-eval', '--load-config', config, '--output-path', '/content/eval.json'], capture_output=True, text=True)
if ev.returncode == 0:
    results = json.load(open('/content/eval.json'))['results']
    print({k: round(v, 3) for k, v in results.items() if not k.endswith('_std')})
else:
    print('Evaluation failed (the splat can still be exported):', ev.stderr[-1500:])

In [ ]:
# 6. Export the splat
import shutil
subprocess.run(['ns-export', 'gaussian-splat', '--load-config', config, '--output-dir', '/content/export'], check=True)
shutil.copy(os.path.join(os.path.dirname(config), 'dataparser_transforms.json'), '/content/export/')
if os.path.exists('/content/eval.json'):
    shutil.copy('/content/eval.json', '/content/export/')
!ls -la /content/export

In [ ]:
# 7. Copy to Google Drive -> open Drive, folder "namma_space_hostel_splat", share it ("Anyone with the link")
from google.colab import drive
drive.mount('/content/drive')
dest = '/content/drive/MyDrive/namma_space_hostel_splat'
os.makedirs(dest, exist_ok=True)
for f in ('splat.ply', 'dataparser_transforms.json', 'eval.json', ):
    if os.path.exists(os.path.join('/content/export', f)):
        shutil.copy(os.path.join('/content/export', f), dest)
print('Saved to Google Drive:', dest)

### Next
Send the Drive link to whoever maintains the repo. The splat is imported with:
```
python -m navkit build app/scenes/hostel      # with "splatSource": "trained_splat.ply" in scene.json
```
The hostel then offers **Points** and a photoreal **Surfaces** view; labels, search and routes stay unchanged.

**No Colab GPU available?** Colab's free GPU time runs out after a few hours of use and comes back later
(usually within a day). Alternatives: **Kaggle Notebooks** (free, 30 GPU hours a week, phone-verified account:
upload this notebook, set *Accelerator → GPU T4*, turn *Internet* on, skip step 7 and download
`/content/export/splat.ply` with `!cp -r /content/export /kaggle/working/` → *Output* panel), or any laptop/PC with an NVIDIA GPU
running Nerfstudio 1.1.5 with the same commands as steps 4–6.